# 03 – Połączenie meczów z pogodą

Wejście: `data/processed/matches_clean.csv` i `data/external/visual_crossing_weather.csv`.

Wyjście: `data/processed/final_dataset.csv` – zbiór analityczny z nazwami kolumn w formacie snake_case.

In [1]:
from pathlib import Path

import pandas as pd

MATCHES_FILE = Path('../data/processed/matches_clean.csv')
WEATHER_FILE = Path('../data/external/visual_crossing_weather.csv')
FINAL_FILE = Path('../data/processed/final_dataset.csv')

KEY_COLS = ['Division', 'Season', 'HomeTeam', 'AwayTeam']

In [2]:
matches = pd.read_csv(MATCHES_FILE, low_memory=False)
weather = pd.read_csv(WEATHER_FILE)
print(f'Mecze: {matches.shape}, pogoda: {weather.shape}')

Mecze: (36271, 172), pogoda: (36271, 20)


## Usunięcie kolumn z dużą liczbą braków

In [3]:
missing_info = (matches.isnull().mean() * 100).sort_values(ascending=False)
print('Procent brakujących danych w kolumnach:')
missing_info.head(10)

Procent brakujących danych w kolumnach:


1XBH       95.299275
1XBA       95.299275
1XBD       95.299275
BFE>2.5    95.238620
BFE<2.5    95.238620
BFEAHH     95.180723
BFEAHA     95.180723
BFA        95.180723
BFD        95.180723
BFH        95.180723
dtype: float64

In [4]:
# Usuwam kolumny, w których brakuje ponad 10% wartości (głównie kursy bukmacherów dostępne tylko w części sezonów)
threshold = 10.0
columns_to_drop = missing_info[missing_info > threshold].index.tolist()
matches_cleaned = matches.drop(columns=columns_to_drop)

print(f'Usunięto {len(columns_to_drop)} kolumn, w których brakowało ponad {threshold}% danych.')
print(f'Nowe wymiary zbioru: {matches_cleaned.shape}')

Usunięto 127 kolumn, w których brakowało ponad 10.0% danych.
Nowe wymiary zbioru: (36271, 45)


## Połączenie po kluczu meczu

W danym sezonie i lidze para gospodarz–gość występuje tylko raz, więc `Division + Season + HomeTeam + AwayTeam` jednoznacznie identyfikuje mecz.

In [5]:
matches_with_weather = matches_cleaned.merge(weather, on=KEY_COLS, how='left', validate='one_to_one')

assert len(matches_with_weather) == len(matches_cleaned)
print(f"Mecze bez danych pogodowych: {matches_with_weather['temp'].isnull().sum()}")
print(f"Mecze z pogodą pobraną dla błędnej daty: {matches_with_weather['date_was_swapped'].sum()}")

Mecze bez danych pogodowych: 165
Mecze z pogodą pobraną dla błędnej daty: 12562


## Nazwy kolumn

In [6]:
matches_with_weather = matches_with_weather.rename(columns={
    'Division': 'division',
    'Season': 'season',
    'HomeTeam': 'home_team',
    'AwayTeam': 'away_team',
    'City': 'city',
    'Stadium': 'stadium',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Date': 'date',
    'Time': 'time',
    'FTHG': 'full_time_home_team_goals',
    'FTAG': 'full_time_away_team_goals',
    'FTR': 'full_time_result',
    'HTHG': 'half_time_home_team_goals',
    'HTAG': 'half_time_away_team_goals',
    'HTR': 'half_time_result',
    'HS': 'home_team_shoots',
    'AS': 'away_team_shoots',
    'HST': 'home_team_shoots_on_target',
    'AST': 'away_team_shoots_on_target',
    'HF': 'home_team_fouls_commited',
    'AF': 'away_team_fouls_commited',
    'HC': 'home_team_corners',
    'AC': 'away_team_corners',
    'HY': 'home_team_yellow_cards',
    'AY': 'away_team_yellow_cards',
    'HR': 'home_team_red_cards',
    'AR': 'away_team_red_cards',
    'B365H': 'bet365_home_win_odds',
    'B365D': 'bet365_draw_odds',
    'B365A': 'bet365_away_win_odds',
    'BWH': 'betfair_home_win_odds',
    'BWD': 'betfair_draw_odds',
    'BWA': 'betfair_away_win_odds',
    'IWH': 'interwetten_home_win_odds',
    'IWD': 'interwetten_draw_odds',
    'IWA': 'interwetten_away_win_odds',
    'WHH': 'william_hill_home_win_odds',
    'WHD': 'william_hill_draw_odds',
    'WHA': 'william_hill_away_win_odds',
    'VCH': 'vc_bet_home_win_odds',
    'VCD': 'vc_bet_draw_odds',
    'VCA': 'vc_bet_away_win_odds',
    'DayOfWeek': 'day_of_week',
    'Hour': 'hour',
})

print(matches_with_weather.columns.tolist())

['division', 'season', 'home_team', 'away_team', 'city', 'stadium', 'latitude', 'longitude', 'date', 'time', 'full_time_home_team_goals', 'full_time_away_team_goals', 'full_time_result', 'half_time_home_team_goals', 'half_time_away_team_goals', 'half_time_result', 'home_team_shoots', 'away_team_shoots', 'home_team_fouls_commited', 'away_team_fouls_commited', 'home_team_corners', 'away_team_corners', 'home_team_yellow_cards', 'away_team_yellow_cards', 'home_team_red_cards', 'away_team_red_cards', 'bet365_home_win_odds', 'bet365_draw_odds', 'bet365_away_win_odds', 'betfair_home_win_odds', 'betfair_draw_odds', 'betfair_away_win_odds', 'interwetten_home_win_odds', 'interwetten_draw_odds', 'interwetten_away_win_odds', 'william_hill_home_win_odds', 'william_hill_draw_odds', 'william_hill_away_win_odds', 'vc_bet_home_win_odds', 'vc_bet_draw_odds', 'vc_bet_away_win_odds', 'home_team_shoots_on_target', 'away_team_shoots_on_target', 'day_of_week', 'hour', 'query_date', 'query_time', 'date_was_sw

In [7]:
matches_with_weather.to_csv(FINAL_FILE, index=False)
print(f'Zapisano {FINAL_FILE}, wymiary: {matches_with_weather.shape}')

Zapisano ../data/processed/final_dataset.csv, wymiary: (36271, 61)
